
# Seção 3 — Arquitetura Híbrida com API

Este notebook implementa a arquitetura principal proposta no TCC:

**texto do projeto → LLM → evento estruturado → grafo → propagação de impacto → regras documentais → rastreabilidade**

## Objetivo

Avaliar o comportamento da arquitetura híbrida usando os mesmos cenários do baseline.

A diferença é:

### Baseline
**texto → LLM → resposta final**

### Arquitetura proposta
**texto → LLM → evento estruturado → grafo + regras → resposta final**

Assim, o LLM fica responsável principalmente por **interpretar a linguagem natural**, enquanto o grafo e as regras fazem parte do raciocínio estruturado.

## Verificações antes do experimento

Eventos `informational` não propagam impactos, mesmo quando citam entidades.
A interpretação é validada antes do pipeline: lista de IDs, pessoa responsável,
datas no formato YYYY-MM-DD e confiança numérica entre 0 e 1.
Respostas inválidas preservam texto bruto, ID da resposta e etapa do erro.
A avaliação exige exatamente um registro por cenário de referência, sem IDs
faltantes, desconhecidos ou duplicados. Tentativas com erro continuam no conjunto
e as contagens de falhas são exibidas nas métricas.

Os testes locais usam respostas simuladas; não medem o desempenho do LLM.
A chave fica no `.env` da raiz ou no ambiente local, nunca no notebook.
A configuração carrega o `.env` sem exibir seu conteúdo. Não há dependência de Drive ou Colab.


## 30. Localizar o repositório


In [ ]:
from pathlib import Path
import sys

# Descobre a raiz a partir da pasta local, sem caminhos pessoais fixos.
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "TCC_Codigos/data/project_graph.json").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Abra o projeto a partir da raiz do repositório TCC.")
PROJECT_DIR = REPO_ROOT / "TCC_Codigos"
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Projeto local:", PROJECT_DIR)


## 31. Preparar ambiente

In [ ]:
from datetime import datetime, timezone
from pprint import pprint
import hashlib
import json
import os
import re
import time

import pandas as pd

print("Ambiente local preparado.")


## 32. Configurar API

In [ ]:
from openai import OpenAI
from tcc_agent.local_runtime import load_local_env

load_local_env(REPO_ROOT)
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Configure OPENAI_API_KEY no .env da raiz ou no ambiente local.")
client = OpenAI(api_key=api_key)
print("Cliente configurado a partir do ambiente local. Chave não exibida.")


## 33. Carregar componentes da arquitetura

In [ ]:

from tcc_agent.models import Event
from tcc_agent.graph import ProjectGraph
from tcc_agent.rules import artifacts_for_event, EVENT_TO_ARTIFACTS
from tcc_agent.pipeline import HybridImpactPipeline

print("Componentes importados com sucesso.")


## 34. Carregar Projeto Alfa e cenários

In [ ]:

graph_file = PROJECT_DIR / "data" / "project_graph.json"
scenarios_file = PROJECT_DIR / "data" / "scenarios_ground_truth.json"

graph = ProjectGraph.from_json(graph_file)

with open(graph_file, "r", encoding="utf-8") as f:
    project_graph_raw = json.load(f)

with open(scenarios_file, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

entity_catalog = [
    {
        "id": node["id"],
        "name": node["name"],
        "type": node["type"],
    }
    for node in project_graph_raw["nodes"]
]

print("Entidades no grafo:", len(graph.nodes))
print("Relações no grafo:", len(graph.edges))
print("Cenários:", len(scenarios))

with open(PROJECT_DIR / "data" / "scenarios_development_pilot.json", "r", encoding="utf-8") as f:
    pilot_scenarios = json.load(f)



## 35. Definir modelo da arquitetura híbrida

Use o mesmo modelo escolhido para o baseline.

Isso é importante para que a comparação seja justa: a variável que queremos testar é a **arquitetura**, não trocar o modelo.


In [ ]:

MODEL = "gpt-6-luna"

ALLOWED_EVENT_TYPES = [
    "delay",
    "decision",
    "test_problem",
    "technical_change",
    "action_commitment",
    "risk_signal",
    "supplier_change",
    "informational",
]

print("Modelo:", MODEL)



## 36. Prompt da etapa de interpretação

Aqui o LLM **não precisa prever todos os impactos do projeto**.

Ele deve apenas transformar linguagem natural em uma representação estruturada do evento.

Isso é uma diferença importante em relação ao baseline.


In [ ]:

HYBRID_INTERPRETER_PROMPT_V1 = """
Você é o módulo de interpretação semântica de um sistema de gestão de projetos.

Sua tarefa é analisar UM evento textual e transformar esse texto em um evento estruturado.

Você NÃO deve:
- tentar reconstruir toda a cadeia de impacto do projeto;
- inventar dependências;
- decidir sozinho todos os documentos afetados.

Essas tarefas serão feitas depois pelo grafo e pelas regras do sistema.

Utilize somente IDs existentes no catálogo de entidades.

Tipos de evento permitidos:
delay
decision
test_problem
technical_change
action_commitment
risk_signal
supplier_change
informational

Retorne APENAS um JSON válido neste formato:

{
  "event_type": "string",
  "primary_entity": "ID ou null",
  "related_entities": ["ID"],
  "responsible": "ID ou null",
  "due_date": "YYYY-MM-DD ou null",
  "effective_date": "YYYY-MM-DD ou null",
  "confidence": 0.0,
  "rationale": "justificativa curta"
}

Regras:
- confidence deve ficar entre 0 e 1;
- não invente IDs;
- se o evento não gerar impacto gerencial, use informational;
- related_entities deve conter apenas elementos explicitamente relacionados ao evento.
""".strip()

hybrid_prompt_hash = hashlib.sha256(
    HYBRID_INTERPRETER_PROMPT_V1.encode("utf-8")
).hexdigest()[:16]

print("Prompt: HYBRID_INTERPRETER_V1")
print("Hash:", hybrid_prompt_hash)


## 37. Criar funções de interpretação com LLM

In [ ]:

def build_interpreter_input(scenario):
    catalog_text = "\n".join(
        f"- {e['id']} | {e['type']} | {e['name']}"
        for e in entity_catalog
    )

    return f"""
CENÁRIO
ID: {scenario['scenario_id']}
Data: {scenario.get('occurred_at')}
Fonte: {scenario.get('source_type')}
Texto: {scenario['source_text']}

CATÁLOGO DE ENTIDADES VÁLIDAS
{catalog_text}

Transforme o texto em um evento estruturado.
""".strip()


def extract_json(text):
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


def validate_interpretation(pred):
    from datetime import date
    import math

    if not isinstance(pred, dict):
        raise ValueError("A interpretação deve ser um objeto JSON.")

    required = {
        "event_type", "primary_entity", "related_entities", "responsible",
        "due_date", "effective_date", "confidence", "rationale",
    }
    missing = required - set(pred)
    if missing:
        raise ValueError(f"Campos ausentes: {sorted(missing)}")

    event_type = pred["event_type"]
    if not isinstance(event_type, str) or event_type not in ALLOWED_EVENT_TYPES:
        raise ValueError("event_type inválido.")

    entities = {e["id"]: e for e in entity_catalog}

    def validate_entity(value, field):
        if value is not None and (
            not isinstance(value, str) or value not in entities
        ):
            raise ValueError(f"{field} deve ser um ID válido ou null.")

    validate_entity(pred["primary_entity"], "primary_entity")
    related = pred["related_entities"]
    if not isinstance(related, list):
        raise ValueError("related_entities deve ser uma lista de IDs, nunca null.")
    for value in related:
        if not isinstance(value, str) or value not in entities:
            raise ValueError("related_entities contém um ID inválido.")

    responsible = pred["responsible"]
    validate_entity(responsible, "responsible")
    if responsible is not None and entities[responsible]["type"] != "Person":
        raise ValueError("responsible deve identificar uma pessoa (Person).")

    for field in ("due_date", "effective_date"):
        value = pred[field]
        if value is None:
            continue
        if not isinstance(value, str):
            raise ValueError(f"{field} deve usar YYYY-MM-DD ou null.")
        try:
            parsed = date.fromisoformat(value)
        except ValueError as exc:
            raise ValueError(f"{field} contém uma data inválida.") from exc
        if parsed.isoformat() != value:
            raise ValueError(f"{field} deve usar exatamente YYYY-MM-DD.")

    confidence = pred["confidence"]
    if (
        isinstance(confidence, bool)
        or not isinstance(confidence, (int, float))
        or not math.isfinite(confidence)
        or not 0 <= confidence <= 1
    ):
        raise ValueError("confidence deve ser um número finito entre 0 e 1.")
    if not isinstance(pred["rationale"], str) or not pred["rationale"].strip():
        raise ValueError("rationale deve ser uma justificativa textual não vazia.")

    return True


def call_interpreter_llm(scenario):
    started = time.time()
    response = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": HYBRID_INTERPRETER_PROMPT_V1},
            {"role": "user", "content": build_interpreter_input(scenario)},
        ],
    )

    # Preserve a resposta antes de interpretar ou validar seu conteúdo.
    result = {
        "prediction": None,
        "raw_output": response.output_text,
        "response_id": getattr(response, "id", None),
        "elapsed_seconds": round(time.time() - started, 3),
        "error": None,
        "error_stage": None,
        "error_type": None,
    }
    stage = "parse"
    try:
        pred = extract_json(result["raw_output"])
        stage = "validation"
        validate_interpretation(pred)
        result["prediction"] = pred
    except Exception as exc:
        result["error"] = str(exc)
        result["error_stage"] = stage
        result["error_type"] = type(exc).__name__

    return result


## 38. Converter saída do LLM em Event

In [ ]:

def prediction_to_event(prediction, evidence):
    validate_interpretation(prediction)
    return Event(
        event_type=prediction["event_type"],
        primary_entity=prediction.get("primary_entity"),
        related_entities=prediction.get("related_entities", []),
        responsible=prediction.get("responsible"),
        due_date=prediction.get("due_date"),
        effective_date=prediction.get("effective_date"),
        evidence=evidence,
        confidence=prediction.get("confidence"),
    )

print("Conversor LLM -> Event criado.")


## 39. Criar o pipeline híbrido

In [ ]:

pipeline = HybridImpactPipeline(
    graph=graph,
    max_depth=4
)

print("Pipeline híbrido criado.")



## 40. Criar função de execução completa

Esta é a principal função da arquitetura:

**texto → LLM → Event → Pipeline → resultado**


In [ ]:

def run_hybrid_scenario(scenario):
    started_total = time.time()

    record = {
        "scenario_id": scenario["scenario_id"],
        "source_text": scenario["source_text"],
        "model": MODEL,
        "prompt_version": "HYBRID_INTERPRETER_V1",
        "prompt_hash": hybrid_prompt_hash,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "interpretation_ok": False,
        "pipeline_ok": False,
        "llm_prediction": None,
        "impacted_entities": [],
        "affected_artifacts": [],
        "trace": {},
        "raw_output": None,
        "response_id": None,
        "llm_elapsed_seconds": None,
        "error_stage": None,
        "error_type": None,
        "error": None,
    }

    stage = "api"
    try:
        llm_result = call_interpreter_llm(scenario)
        record["raw_output"] = llm_result["raw_output"]
        record["response_id"] = llm_result["response_id"]
        record["llm_elapsed_seconds"] = llm_result["elapsed_seconds"]

        if llm_result["error"] is not None:
            record["error"] = llm_result["error"]
            record["error_stage"] = llm_result["error_stage"]
            record["error_type"] = llm_result["error_type"]
        else:
            record["llm_prediction"] = llm_result["prediction"]
            record["interpretation_ok"] = True
            stage = "pipeline"
            event = prediction_to_event(
                llm_result["prediction"], evidence=scenario["source_text"]
            )
            result = pipeline.analyze(event)
            record["impacted_entities"] = list(result.impacted_entities)
            record["affected_artifacts"] = list(result.affected_artifacts)
            record["trace"] = dict(result.trace)
            record["pipeline_ok"] = True

    except Exception as exc:
        record["error"] = str(exc)
        record["error_stage"] = stage
        record["error_type"] = type(exc).__name__

    record["total_elapsed_seconds"] = round(
        time.time() - started_total,
        3
    )

    return record


## 41. Testar DEV001 de ponta a ponta

In [ ]:

pilot_example = next(
    s for s in pilot_scenarios
    if s["scenario_id"] == "DEV001"
)

pilot_result = run_hybrid_scenario(pilot_example)

pprint(pilot_result)


## 42. Visualização amigável do resultado

In [ ]:

def show_hybrid_result(record):
    print("=" * 75)
    print("CENÁRIO:", record["scenario_id"])
    print("=" * 75)

    print("\nTEXTO")
    print(record["source_text"])

    print("\nINTERPRETAÇÃO DO LLM")
    pred = record.get("llm_prediction")

    if pred:
        print("Tipo:", pred.get("event_type"))
        print("Entidade principal:", pred.get("primary_entity"))
        print("Relacionadas:", pred.get("related_entities"))
        print("Confiança:", pred.get("confidence"))
        print("Justificativa:", pred.get("rationale"))
    else:
        print("Não disponível.")

    print("\nDOCUMENTOS A REVISAR")
    if record["affected_artifacts"]:
        for artifact in record["affected_artifacts"]:
            print("-", artifact)
    else:
        print("- Nenhum")

    print("\nENTIDADES IMPACTADAS")
    if record["impacted_entities"]:
        for entity_id in record["impacted_entities"]:
            print("-", entity_id, "-", graph.entity_name(entity_id))
    else:
        print("- Nenhuma")

    print("\nRASTREABILIDADE")
    if record["trace"]:
        for target, path in record["trace"].items():
            names = [graph.entity_name(x) for x in path]
            print("-", " -> ".join(names))
    else:
        print("- Nenhum caminho")

    if record["error"]:
        print("\nERRO")
        print(record["error"])

show_hybrid_result(result_001)


## 43. Executar piloto com 6 cenários
O piloto usa exclusivamente `DEV001`–`DEV006`; os 36 casos oficiais ficam reservados ao teste após congelamento.


In [ ]:

PILOT_IDS = ["DEV001", "DEV002", "DEV003", "DEV004", "DEV005", "DEV006"]

hybrid_pilot = []

for sid in PILOT_IDS:
    scenario = next(s for s in pilot_scenarios if s["scenario_id"] == sid)

    print("Executando:", sid)

    record = run_hybrid_scenario(scenario)
    hybrid_pilot.append(record)

    print(
        "  interpretation_ok =", record["interpretation_ok"],
        "| pipeline_ok =", record["pipeline_ok"],
        "| error =", record["error"]
    )

print("\nPiloto híbrido concluído.")


## 44. Salvar piloto
O piloto usa exclusivamente `DEV001`–`DEV006`; os 36 casos oficiais ficam reservados ao teste após congelamento.


In [ ]:

OUTPUT_DIR = PROJECT_DIR / "outputs" / "hybrid_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(
    OUTPUT_DIR / "hybrid_pilot_v1.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for record in hybrid_pilot:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

(OUTPUT_DIR / "hybrid_interpreter_prompt_v1.txt").write_text(
    HYBRID_INTERPRETER_PROMPT_V1,
    encoding="utf-8"
)

metadata = {
    "model": MODEL,
    "prompt_version": "HYBRID_INTERPRETER_V1",
    "prompt_hash": hybrid_prompt_hash,
    "max_depth": 4,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

with open(
    OUTPUT_DIR / "metadata_v1.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Resultados salvos em:")
print(OUTPUT_DIR)



## 45. Congelar a arquitetura antes dos 36 casos

Antes de executar os 36 cenários:

- [ ] modelo igual ao baseline;
- [ ] prompt do interpretador congelado;
- [ ] profundidade do grafo congelada;
- [ ] regras documentais congeladas;
- [ ] ground truth congelado;
- [ ] piloto revisado;
- [ ] nenhuma resposta esperada está sendo enviada ao LLM.

Depois disso, altere:

`RUN_FULL_HYBRID = True`

O piloto usa exclusivamente `DEV001`–`DEV006`; os 36 casos oficiais ficam reservados ao teste após congelamento.


In [ ]:

RUN_FULL_HYBRID = False

print("RUN_FULL_HYBRID =", RUN_FULL_HYBRID)


## 46. Executar os 36 cenários com a arquitetura híbrida

In [ ]:

hybrid_results_file = OUTPUT_DIR / "hybrid_full_results_v1.jsonl"

if RUN_FULL_HYBRID:
    hybrid_full_records = []

    hybrid_results_file.write_text("", encoding="utf-8")

    for idx, scenario in enumerate(scenarios, start=1):
        sid = scenario["scenario_id"]

        print(
            f"[{idx:02d}/{len(scenarios)}] "
            f"Executando {sid}..."
        )

        record = run_hybrid_scenario(scenario)
        hybrid_full_records.append(record)

        with open(
            hybrid_results_file,
            "a",
            encoding="utf-8"
        ) as f:
            f.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )

        time.sleep(0.25)

    print("\nExecução concluída.")
else:
    print(
        "Experimento completo ainda bloqueado. "
        "Altere RUN_FULL_HYBRID para True somente após o congelamento."
    )


## 47. Carregar resultados salvos

In [ ]:

if (
    hybrid_results_file.exists()
    and hybrid_results_file.stat().st_size > 0
):
    with open(
        hybrid_results_file,
        "r",
        encoding="utf-8"
    ) as f:
        hybrid_full_records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    print("Resultados carregados:", len(hybrid_full_records))
else:
    hybrid_full_records = []
    print("Ainda não existem resultados completos.")


## 48. Avaliar arquitetura híbrida

In [ ]:

def safe_set(values):
    return set(values or [])


def validate_result_coverage(records, scenarios):
    from collections import Counter

    if not scenarios:
        raise ValueError("Não há cenários de referência para avaliar.")
    expected_ids = [s["scenario_id"] for s in scenarios]
    if len(set(expected_ids)) != len(expected_ids):
        raise ValueError("Há IDs duplicados no ground truth.")
    if any(
        not isinstance(r, dict) or not isinstance(r.get("scenario_id"), str)
        for r in records
    ):
        raise ValueError("Todo resultado deve conter um scenario_id textual.")
    counts = Counter(r["scenario_id"] for r in records)
    expected = set(expected_ids)
    missing = sorted(expected - set(counts))
    unknown = sorted(set(counts) - expected)
    duplicates = sorted(sid for sid, count in counts.items() if count > 1)
    if missing or unknown or duplicates:
        raise ValueError(
            "Cobertura inválida: "
            f"ausentes={missing}; desconhecidos={unknown}; duplicados={duplicates}. "
            "A avaliação requer exatamente um resultado por cenário, "
            "incluindo registros de tentativas com erro."
        )


def evaluate_hybrid(records, scenarios):
    validate_result_coverage(records, scenarios)
    gt_by_id = {
        s["scenario_id"]: s
        for s in scenarios
    }

    rows = []

    event_correct = 0
    event_total = 0

    artifact_tp = 0
    artifact_fp = 0
    artifact_fn = 0

    entity_tp = 0
    entity_fp = 0
    entity_fn = 0

    for record in records:
        sid = record["scenario_id"]
        gt = gt_by_id[sid]

        pred = record.get("llm_prediction")

        event_total += 1

        event_ok = (
            pred is not None
            and pred.get("event_type")
            == gt["expected_event_type"]
        )

        if event_ok:
            event_correct += 1

        expected_artifacts = safe_set(
            gt["expected_artifacts"]
        )
        predicted_artifacts = safe_set(
            record.get("affected_artifacts")
        )

        expected_entities = safe_set(
            gt["expected_impacted_entities"]
        )
        predicted_entities = safe_set(
            record.get("impacted_entities")
        )

        atp = len(
            expected_artifacts & predicted_artifacts
        )
        afp = len(
            predicted_artifacts - expected_artifacts
        )
        afn = len(
            expected_artifacts - predicted_artifacts
        )

        etp = len(
            expected_entities & predicted_entities
        )
        efp = len(
            predicted_entities - expected_entities
        )
        efn = len(
            expected_entities - predicted_entities
        )

        artifact_tp += atp
        artifact_fp += afp
        artifact_fn += afn

        entity_tp += etp
        entity_fp += efp
        entity_fn += efn

        ap = atp / (atp + afp) if atp + afp else (
            1.0 if not expected_artifacts else 0.0
        )
        ar = atp / (atp + afn) if atp + afn else 1.0
        af1 = (
            2 * ap * ar / (ap + ar)
            if ap + ar else 0.0
        )

        ep = etp / (etp + efp) if etp + efp else (
            1.0 if not expected_entities else 0.0
        )
        er = etp / (etp + efn) if etp + efn else 1.0
        ef1 = (
            2 * ep * er / (ep + er)
            if ep + er else 0.0
        )

        rows.append({
            "scenario_id": sid,
            "event_correct": event_ok,
            "artifact_precision": round(ap, 4),
            "artifact_recall": round(ar, 4),
            "artifact_f1": round(af1, 4),
            "entity_precision": round(ep, 4),
            "entity_recall": round(er, 4),
            "entity_f1": round(ef1, 4),
            "error": record.get("error"),
        })

    event_accuracy = (
        event_correct / event_total
        if event_total else 0.0
    )

    artifact_precision = (
        artifact_tp / (artifact_tp + artifact_fp)
        if artifact_tp + artifact_fp else 0.0
    )
    artifact_recall = (
        artifact_tp / (artifact_tp + artifact_fn)
        if artifact_tp + artifact_fn else 0.0
    )
    artifact_f1 = (
        2 * artifact_precision * artifact_recall
        / (artifact_precision + artifact_recall)
        if artifact_precision + artifact_recall else 0.0
    )

    entity_precision = (
        entity_tp / (entity_tp + entity_fp)
        if entity_tp + entity_fp else 0.0
    )
    entity_recall = (
        entity_tp / (entity_tp + entity_fn)
        if entity_tp + entity_fn else 0.0
    )
    entity_f1 = (
        2 * entity_precision * entity_recall
        / (entity_precision + entity_recall)
        if entity_precision + entity_recall else 0.0
    )

    summary = {
        "n_scenarios": event_total,
        "n_expected_scenarios": len(scenarios),
        "n_pipeline_failures": sum(not r.get("pipeline_ok", False) for r in records),
        "n_interpretation_failures": sum(not r.get("interpretation_ok", False) for r in records),
        "event_type_accuracy": event_accuracy,
        "artifact_precision_micro": artifact_precision,
        "artifact_recall_micro": artifact_recall,
        "artifact_f1_micro": artifact_f1,
        "entity_precision_micro": entity_precision,
        "entity_recall_micro": entity_recall,
        "entity_f1_micro": entity_f1,
    }

    return summary, pd.DataFrame(rows)


if hybrid_full_records:
    hybrid_summary, hybrid_details = evaluate_hybrid(
        hybrid_full_records,
        scenarios
    )

    print("RESULTADOS DA ARQUITETURA HÍBRIDA\n")

    for key, value in hybrid_summary.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    display(hybrid_details)
else:
    print("Execute os 36 casos antes da avaliação.")


## 49. Salvar métricas

In [ ]:

if hybrid_full_records:
    with open(
        OUTPUT_DIR / "hybrid_metrics_v1.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            hybrid_summary,
            f,
            ensure_ascii=False,
            indent=2
        )

    hybrid_details.to_csv(
        OUTPUT_DIR / "hybrid_details_v1.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Métricas salvas.")
else:
    print("Ainda não existem métricas para salvar.")



# O que será comparado com o baseline

Depois que os dois experimentos forem executados, teremos:

| Condição | Interpretação | Impacto | Documentos |
|---|---|---|---|
| Baseline | LLM | LLM | LLM |
| Arquitetura híbrida | LLM | Grafo | Regras |

A comparação principal será feita com:

- acurácia do tipo de evento;
- precision;
- recall;
- F1;
- falsos positivos;
- falsos negativos;
- rastreabilidade.

## Hipótese do TCC

A arquitetura híbrida deverá produzir impactos mais precisos e mais rastreáveis que o uso isolado do LLM.

Essa hipótese só poderá ser confirmada ou rejeitada depois da execução dos experimentos.
